In [2]:
import mne
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
edf_path = r'D:\2025_26_internship\bigP3BCI_pipeline\Data\StudyA\A_01\SE001\Train\CB\A_01_SE001_CB_Train01.edf'


# Load the EDF file
raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

# Print info to verify
print(raw.info)
print(raw.annotations)  # Check stimuli markers; P300 datasets use event codes for row/column flashes

# Select only EEG channels (exclude EOG, etc., if present)
eeg_picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False)
raw.pick(picks=eeg_picks)

<Info | 8 non-empty values
 bads: []
 ch_names: EEG_F3, EEG_Fz, EEG_F4, EEG_T7, EEG_C3, EEG_Cz, EEG_C4, EEG_T8, ...
 chs: 114 EEG
 custom_ref_applied: False
 highpass: 58.0 Hz
 lowpass: 62.0 Hz
 meas_date: 2020-01-01 00:00:00 UTC
 nchan: 114
 projs: []
 sfreq: 256.0 Hz
 subject_info: <subject_info | his_id: A_01, sex: 0, first_name: X, last_name: X, birthday: 2020-01-01>
>
<Annotations | 1 segment: Begin recording (1)>


C:\Users\qigan\AppData\Local\Temp\ipykernel_22564\1332155822.py:10: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
C:\Users\qigan\AppData\Local\Temp\ipykernel_22564\1332155822.py:10: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


<RawEDF | A_01_SE001_CB_Train01.edf, 114 x 35208 (137.5 s), ~30.7 MiB, data loaded>

In [2]:
# Bandpass filter
raw.filter(l_freq=0.5, h_freq=30.0, fir_design='firwin')

# Basic artifact removal: Reject epochs with amplitude > 100 uV (adjust based on data)
reject_criteria = dict(eeg=100e-6)  # 100 uV

# Create events from annotations (assuming 'StimulusBegin' or similar; adapt to your file)
events, event_id = mne.events_from_annotations(raw)
# Example event_id: {'non-target': 1, 'target': 2} – map based on dataset docs

# Epoching: -0.2 to 0.8 s around stimuli
epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=0.8,
                    baseline=(-0.2, 0), reject=reject_criteria, preload=True)

# Drop bad epochs
epochs.drop_bad()

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1691 samples (6.605 s)

Used Annotations descriptions: [np.str_('Begin recording')]
Not setting metadata
1 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 1 events and 257 original time points ...
1 bad epochs dropped


C:\Users\qigan\AppData\Local\Temp\ipykernel_42128\3479078934.py:12: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=0.8,


In [4]:
event_id

{np.str_('Begin recording'): 1}